# 📗 Cypher 심화: 경로 탐색

지난 시간에는 관계를 **칸 수만큼 손으로 적어** 이어 붙였습니다(`(a)-[:R]->(b)-[:S]->(c)`). 그런데 지하철에서 "**두 정거장 안에 갈 수 있는 역**"이나 "**여기서 저기까지 가장 빠른 길**"을 물으면, 관계를 몇 번 타야 할지 미리 알 수 없어 손으로 적을 수가 없습니다.

이번 시간에는 관계를 **여러 칸 이어서** 따라가는 법을 배웁니다. **가변길이 패턴**(`*1..2`)으로 몇 정거장 안의 역을 찾고, **`shortestPath`** 로 최단 경로를 구하고, **`length`·`nodes`** 로 그 경로를 해부하고, **리스트 표현식**으로 목록에서 값만 뽑은 뒤, 그 **경로 위 구간에 조건을 겁니다**.

데모는 **수도권 전철**(실제 노선 자료, 역 659곳), 따라하기는 **계좌 송금**입니다.

## ⏪ 복습: 지난 시간까지

- **CREATE·MATCH·RETURN**: 노드와 관계를 만들고, 패턴으로 조회했습니다.
- **WHERE·MERGE**: 조건으로 거르고, 멱등하게 만들었습니다.
- **DISTINCT·ORDER BY·LIMIT**: 되풀이를 접고, 줄을 세우고, 위에서 몇 줄만 받았습니다. 오늘 1-1 과 3-3 에서 그대로 다시 씁니다.
- **관계 패턴**: `(a)-[:R]->(b)` 처럼 방향과 관계 종류를 그림처럼 적었습니다. 오늘은 이 관계를 **여러 칸 이어서** 따라갑니다.

**오늘의 목표**

**1. 가변길이 패턴**
- [ ] (1-1) **가변길이 패턴**(`*1..2`)으로 N 칸 안에 닿는 노드를 찾고 `DISTINCT` 로 중복을 없앤다. 범위를 여는 **여러 표기**(`*2..2`·`*1..`·`*`·`*..2`·`*0..2`)도 구분해 쓴다.

**2. 최단 경로**
- [ ] (2-1) **`shortestPath`** 로 두 노드 사이 최단 경로를 구한다.
- [ ] (2-2) 최단 경로가 여럿일 때 **`allShortestPaths`** 로 전부 받는다.

**3. 경로 해부와 조건**
- [ ] (3-1) **경로 변수**(`p=...`)를 잡아 **`length`·`nodes`·`relationships`** 로 해부한다.
- [ ] (3-2) **리스트 표현식**(`[n IN 목록 | 표현식]`)으로 목록에서 값만 뽑는다.
- [ ] (3-3) **`length(p)`** 로 줄을 세워 가장 먼 곳을 찾는다.
- [ ] (3-4) **`all`·`any`·`none`** 으로 경로 위 구간에 조건을 건다.

아래 준비 셀을 먼저 실행하세요. **연결 → 전철 그래프 적재 → 송금 그래프 적재** 순서입니다. 전철 그래프는 역이 많아 적재에 몇 초 걸립니다.

> ⚠️ **연결 셀이 오류로 멈춘다면** 아래 셋 중 하나입니다. 오류 메시지 마지막 줄부터 보세요.
>
> 1. `ServiceUnavailable` 또는 `Couldn't connect`: 실습 전용 Neo4j 인스턴스가 **꺼져 있습니다**. Neo4j Desktop 에서 그 인스턴스를 Start 하고 다시 실행하세요.
> 2. `AuthError`: `.env` 의 `NEO4J_PASSWORD` 가 인스턴스 비밀번호와 다릅니다.
> 3. 아무 값도 못 읽는 것 같다면 이 폴더에 **`.env` 파일이 없는** 경우입니다. 같은 폴더의 `.env.example` 을 복사해 `.env` 로 만들고 `NEO4J_URI`·비밀번호를 채우세요.

In [ ]:
# Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

오늘 조회할 그래프는 **수도권 전철**입니다. 역 **659곳**이 구간 **778개**로 이어져 있고 노선은 **25가지**입니다. 아래 그림은 그중 도심 몇 정거장만 잘라 온 도식입니다.

- 역 노드 `:Station` 에는 이름(`name`)과 **그 역을 지나는 노선 목록**(`lines`)이 붙어 있습니다.
- 구간 관계 `:NEXT_TO` 에는 **그 구간이 몇 호선인지**(`line`)와 **두 역 사이 거리**(`km`)가 붙어 있습니다. 거리는 두 역 좌표 사이의 **직선거리**라 실제 선로 길이보다 조금 짧습니다(2-2 에서 이 값으로 길이를 재 봅니다).
- 구간은 저장할 때만 한 방향으로 적어 두었습니다. 실제로는 양쪽으로 오갈 수 있으므로 **조회할 때는 화살표 없이** `-[:NEXT_TO]-` 로 씁니다.

> 이 자료는 노선별로 이웃한 역을 이어 만든 것이라, 급행이나 직결 운행 구간이 한 칸으로 이어진 곳도 있습니다. 그래서 여기서 세는 "정거장 수"는 실제 소요 시간과 정확히 같지는 않습니다.

<img src="images/지하철_그래프.png" width="1000">

*역과 구간이 어떻게 이어지는지 보여 주는 도식입니다(전체 그래프 가운데 도심 구간만 잘라 왔습니다).*

In [ ]:
# 수도권 전철 그래프 적재: 이 셀은 실행만 하세요(5초쯤 걸립니다).
# CSV 두 장(역·구간)을 읽어 그래프를 만듭니다. 자료 출처는 OpenStreetMap 입니다.
# 적재 코드는 길어서 data/load_subway.py 에 따로 두었습니다. 그 폴더를 경로에 넣고 import 합니다.
import sys
from pathlib import Path

# 교안 폴더에서 열면 data/, 정답 폴더에서 열면 ../data/ 가 맞는 자리다
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
if str(DATA_DIR) not in sys.path:      # 셀을 두 번 실행해도 경로가 쌓이지 않게 한다
    sys.path.insert(0, str(DATA_DIR))
from load_subway import load           # data/load_subway.py 의 load 함수

# 두 번 실행해도 역이 두 벌로 늘지 않게, 만들기 전에 지운다
# 지우는 대상을 :Station 으로 좁혔으니 다른 실습 그래프는 건드리지 않는다
run_cypher("MATCH (n:Station) DETACH DELETE n")

# load 는 CSV 를 읽어 적재하고 만든 역·구간 개수를 돌려준다
n_station, n_edge = load(run_cypher)
print("전철 그래프 적재 완료. 역:", n_station, "개, 구간:", n_edge, "개")

전철 그래프에는 노선이 둘 이상 지나는 **환승역이 118곳** 있습니다. 오늘 출발역으로 자주 쓸 서울역도 그중 하나로 `lines` 가 `['1호선', '4호선', 'GTX-A', '경의·중앙선', '공항철도']` 입니다.

데이터를 분석하기 전에 **무엇이 들어 있는지 먼저 훑어봅니다.** 아래 셀은 실행만 하세요.

In [ ]:
# 전철 그래프에 무엇이 들어 있는지 먼저 훑어봅니다(실행만 하세요)
# 1) 역을 전부 받아 센다. run_cypher 가 행 목록을 주므로 len 이 곧 행 수다
stations = run_cypher("MATCH (n:Station) RETURN n.name AS 역")
print('역 수:', len(stations))

In [ ]:
# 2) 구간은 관계다. 저장이 한 방향이라 화살표를 적어 센다
segments = run_cypher("MATCH (a:Station)-[x:NEXT_TO]->(b:Station) RETURN x.line AS 노선")
print('구간 수:', len(segments))

In [ ]:
# 3) 노선은 구간마다 붙어 있다. 집합으로 접으면 종류 수가 나온다
lines = {r['노선'] for r in segments}
print('노선 수:', len(lines))

In [ ]:
# 4) lines 는 노선 '목록'이다. 이어 붙인 문자열이 아니다
seoul = run_cypher("MATCH (n:Station {name:'서울역'}) RETURN n.name AS 역, n.lines AS 노선목록")
print('서울역이 지나는 노선:', seoul[0]['노선목록'])

따라하기에 쓸 **계좌 송금** 그래프입니다. 계좌 `:Account` 를 송금 `:TRANSFER` 가 잇고, 구간마다 보낸 금액(`amount`)과 날짜(`date`)가 붙어 있습니다. 계좌 11개, 송금 14건입니다.

전철과 결정적으로 다른 점이 하나 있습니다. **돈은 한쪽으로만 흐릅니다.** 그래서 따라하기에서는 화살표를 빼지 않고 `-[:TRANSFER*]->` 처럼 **방향을 그대로 둡니다.**

이런 그래프를 두고 실제로 하는 일이 **이상거래 탐지**입니다. "이 계좌에서 나간 돈이 어디까지 흘러갔나", "큰 돈만 따라가면 어디에 닿나", "돌고 돌아 제자리로 오는 자리가 있나" 를 묻습니다. 오늘 배우는 **가변길이·최단 경로·경로 조건**이 그 질문을 그대로 옮긴 것입니다.

<img src="images/송금_그래프.png" width="1100">

*계좌 사이의 송금 흐름입니다. 화살표 방향이 곧 돈이 간 방향입니다.*

In [ ]:
# 계좌 송금 시드 적재: 이 셀도 실행만 하세요(따라하기에서 씁니다).
# 레이블·관계 이름이 달라(Account·TRANSFER) 전철 그래프와 섞이지 않습니다.
# 은행 이름, 사람 이름, 계좌번호는 전부 수업용으로 지어낸 가상 정보입니다.
# 구간 속성은 line 이 아니라 amount(보낸 금액, 원)와 date(보낸 날짜)입니다.
# 두 번 실행해도 계좌가 두 벌로 늘지 않게, 만들기 전에 지웁니다
# 지우는 대상을 :Account 로 좁혔으니 방금 적재한 전철 그래프는 그대로 남습니다.
run_cypher("MATCH (n:Account) DETACH DELETE n")
run_cypher("""
CREATE (a1001:Account {name:'A-1001', owner:'김도윤', bank:'ㄱ은행'}),
       (a1002:Account {name:'A-1002', owner:'박서연', bank:'ㄱ은행'}),
       (a1003:Account {name:'A-1003', owner:'이준호', bank:'ㄴ은행'}),
       (a1004:Account {name:'A-1004', owner:'최유진', bank:'ㄴ은행'}),
       (a1005:Account {name:'A-1005', owner:'정민수', bank:'ㄷ은행'}),
       (a1006:Account {name:'A-1006', owner:'한지우', bank:'ㄷ은행'}),
       (a1007:Account {name:'A-1007', owner:'오세훈', bank:'ㄱ은행'}),
       (a1008:Account {name:'A-1008', owner:'윤채원', bank:'ㄴ은행'}),
       (a1009:Account {name:'A-1009', owner:'강태민', bank:'ㄷ은행'}),
       (a1010:Account {name:'A-1010', owner:'임소라', bank:'ㄱ은행'}),
       (a1011:Account {name:'A-1011', owner:'서준영', bank:'ㄴ은행'})
CREATE (a1001)-[:TRANSFER {amount:8000000, date:'2026-03-02'}]->(a1002),
       (a1002)-[:TRANSFER {amount:5000000, date:'2026-03-03'}]->(a1003),
       (a1003)-[:TRANSFER {amount:4500000, date:'2026-03-05'}]->(a1004),
       (a1004)-[:TRANSFER {amount:4000000, date:'2026-03-08'}]->(a1001),
       (a1002)-[:TRANSFER {amount:2500000, date:'2026-03-04'}]->(a1005),
       (a1005)-[:TRANSFER {amount:1800000, date:'2026-03-06'}]->(a1006),
       (a1003)-[:TRANSFER {amount:900000,  date:'2026-03-07'}]->(a1006),
       (a1005)-[:TRANSFER {amount:600000,  date:'2026-03-06'}]->(a1009),
       (a1006)-[:TRANSFER {amount:2200000, date:'2026-03-09'}]->(a1007),
       (a1006)-[:TRANSFER {amount:1100000, date:'2026-03-12'}]->(a1010),
       (a1004)-[:TRANSFER {amount:1500000, date:'2026-03-10'}]->(a1009),
       (a1009)-[:TRANSFER {amount:1200000, date:'2026-03-12'}]->(a1010),
       (a1010)-[:TRANSFER {amount:700000,  date:'2026-03-13'}]->(a1008),
       (a1007)-[:TRANSFER {amount:3000000, date:'2026-03-11'}]->(a1008)
""")
# run_cypher 는 행 리스트를 주므로 len 이 곧 개수다
print("송금 그래프 적재 완료. 계좌:", len(run_cypher("MATCH (a:Account) RETURN a.name")),
      "개, 송금:", len(run_cypher("MATCH ()-[r:TRANSFER]->() RETURN r")), "건")

> 두 그래프는 **레이블과 관계 이름이 달라**(`Station`·`NEXT_TO` / `Account`·`TRANSFER`) 한 데이터베이스에 같이 있어도 섞이지 않습니다. 위 두 셀은 각각 자기 라벨만 지우고 다시 만들기 때문에 **여러 번 실행해도 그래프가 두 벌로 늘지 않습니다.**

---
# 1. 가변길이 패턴으로 여러 칸 따라가기

관계를 **한 칸**만 따라가는 것은 지난 시간에 했습니다. 여기서는 **몇 칸인지 미리 정하지 않고** 이어서 따라가는 법을 배웁니다. `*` 뒤에 적는 범위 표기와, 같은 역이 여러 번 나올 때 접는 법까지 한자리에서 봅니다.

## 1-1. 몇 칸 안에 어떤 역이 있나

### 왜 필요할까요?
"바로 옆 역"은 관계 한 칸(`-[:NEXT_TO]-`)으로 찾습니다. 하지만 "**두 정거장 안의 역**"은 관계를 한 칸일 수도, 두 칸일 수도 있게 **길이를 열어 둬야** 합니다.

### 문법: `*최소..최대`
관계 대괄호 안에 **`*1..2`** 를 붙이면 "이 관계를 **1번에서 2번까지** 이어서 따라가라"는 뜻입니다.

```text
(a)-[:NEXT_TO*1..2]-(b)
        └ 관계 종류  └ 몇 칸까지 이어서 따라갈지
```

여러 갈래로 이어지는 노선에서는 **같은 역에 이르는 길이 하나가 아닐 수** 있습니다. 그러면 그 역이 경로마다 한 번씩, 즉 **여러 번** 나옵니다. **`RETURN DISTINCT`** 로 역 이름을 중복 없이 받습니다.

이 절의 출발역은 **남부터미널** 입니다. 서울역에서 출발하면 이웃이 일곱 곳이라 `*1..2` 만으로도 **26곳**이 쏟아져 화면에 담기지 않습니다.

### 표기 한눈에: `*` 뒤에 올 수 있는 것
질문마다 열어야 할 범위가 다릅니다. "딱 두 칸"·"두 칸 이내"·"몇 칸이든"·"자기 자신도 포함해서"가 전부 다른 질문이고, `*` 뒤에 무엇을 적느냐로 갈립니다.

| 표기 | 뜻 | 언제 |
|---|---|---|
| `*1..1` | 정확히 한 칸 | `*` 없는 것과 같다 |
| `*2..2` | 정확히 두 칸 | "딱 두 정거장 거리" |
| `*1..3` | 1~3 칸 | "세 정거장 안" |
| `*..2` | `*1..2` 의 줄임 | 하한을 안 적으면 1 이다 |
| `*1..` | 1 칸 이상, 상한 없음 | 상한을 두지 않는다 |
| `*` | `*1..` 과 같다 | 상한을 아예 안 둔다 |
| `*0..2` | **0**~2 칸 | 자기 자신도 답에 넣는다 |

하한 `0` 은 "관계를 한 번도 안 타는 것"까지 허용한다는 뜻이라 **출발 노드 자신**이 결과에 들어옵니다. "이 역과 주변 두 정거장을 **한 목록으로** 뽑아라" 같은 질문에 씁니다.

> ⚠️ **상한을 열어 둘 때는 조건이 있습니다.** `*` 나 `*1..` 로 상한을 안 두려면 **한쪽 끝에 이름을 박고**, 결과로 **도착 노드 이름만**(`RETURN DISTINCT b.name`) 받으세요. 이 모양이면 큰 그래프에서도 빠릅니다. 반대로 `length(p)` 처럼 **경로 값을 함께 꺼내면** 조건에 맞는 경로를 **전부** 만들어 보게 되어 금세 수만 갈래가 됩니다. 그때는 상한을 두거나 `shortestPath`(2-1)로 감싸세요. 실제로 얼마나 차이 나는지는 바로 아래에서 재 봅니다.

<img src="images/가변길이_홉수.png" width="760">

*남부터미널에서 `*1..2` 가 훑는 범위입니다. 파란 역이 1칸, 초록 역이 2칸이고, **강남** 에는 교대로 가도 양재로 가도 2칸이라 두 갈래로 닿습니다.*

In [ ]:
# 남부터미널에서 두 정거장 안에 닿는 역. 방향 없이 이어서 찾는다
# *1..2 는 1칸 또는 2칸 이어서 따라가라는 뜻(바로 옆 역도 답에 든다)
# 관계에 변수 r 을 붙이면 탄 구간이 목록으로 담긴다. 길이가 곧 칸 수다
# 같은 관계를 두 번 탈 수 없어 되짚어 돌아가는 길은 안 센다
rows = run_cypher("""
MATCH (a:Station {name:'남부터미널'})-[r:NEXT_TO*1..2]-(b:Station)
RETURN b.name AS 역, r AS 홉
ORDER BY 역
""")
for r in rows:
    print(r)

> **남부터미널**에서 한 칸은 **교대·양재**, 두 칸은 **강남·고속터미널·매봉·서초·양재시민의숲** 입니다. `홉` 목록의 길이가 곧 **몇 칸을 탔는지**입니다.

눈여겨볼 것이 하나 있습니다. **강남이 두 줄로 나왔습니다.** 두 갈래로 퍼진 길이 강남에서 다시 만나는데, 가변길이 패턴은 **경로마다 한 행**을 돌려주기 때문입니다.

역 이름만 중복 없이 받고 싶으면 **`RETURN DISTINCT`** 를 붙입니다. 한 단어만 다른 두 쿼리를 나란히 돌려 행 수를 견줘 봅니다.

In [ ]:
# 두 쿼리는 DISTINCT 한 단어만 다르다. 행 수를 견준다
raw = run_cypher("""
MATCH (a:Station {name:'남부터미널'})-[:NEXT_TO*1..2]-(b:Station)
RETURN b.name AS 역
ORDER BY 역
""")
# DISTINCT 를 붙이면 경로가 몇 갈래든 역 이름 하나에 한 행만 남는다
uniq = run_cypher("""
MATCH (a:Station {name:'남부터미널'})-[:NEXT_TO*1..2]-(b:Station)
RETURN DISTINCT b.name AS 역
ORDER BY 역
""")
print('DISTINCT 없이:', [r['역'] for r in raw])
print('DISTINCT 붙여:', [r['역'] for r in uniq])

> `DISTINCT` 없이는 8행, 붙이면 7행입니다. 늘어난 한 줄이 강남입니다. **"몇 칸 안에 어떤 역이 있나"** 를 물을 때는 역 하나가 한 번만 나와야 하므로 `DISTINCT` 를 붙입니다.

왜 강남만 두 번 나오는지 아래 셀에서 직접 확인합니다. 거기 처음 나오는 표기 세 가지를 먼저 짚고 갑니다.

| 표기 | 뜻 |
|---|---|
| `MATCH p = (...)-[...]-(...)` | 찾은 **경로 전체**를 변수 `p` 에 담는다 |
| `nodes(p)` · `relationships(p)` | 그 경로가 **지나는 역들** · **지나는 구간들**을 목록으로 꺼낸다 |
| `[n IN 목록 \| n.name]` | 목록의 항목마다 **그 속성만** 뽑아 새 목록을 만든다 |

셋째 줄이 파이썬의 `[n.name for n in 목록]` 과 같은 구실을 합니다. Cypher 는 `for` 대신 **`IN`** 을 쓰고, 무엇을 뽑을지는 **세로줄 `|`** 뒤에 적습니다. 그래서 `[n IN nodes(p) | n.name]` 은 "경로가 지나는 역들에서 이름만 뽑아 목록으로" 라는 뜻입니다.

이 셋은 **3절에서 제대로 배웁니다.** 지금은 결과를 읽는 데 필요한 만큼만 알고 넘어가세요.

In [ ]:
# 강남이 두 번 나온 까닭: 남부터미널에서 강남까지 딱 2칸짜리 길을 모두 펼쳐 본다
# p = 는 경로 전체를 변수에 담는다. 거기서 역과 구간 노선을 꺼낸다
rows = run_cypher("""
MATCH p = (a:Station {name:'남부터미널'})-[:NEXT_TO*2..2]-(b:Station {name:'강남'})
RETURN [n IN nodes(p) | n.name] AS 경로,
       [r IN relationships(p) | r.line] AS 구간노선
""")

# 역 사이마다 그 구간의 값을 끼워 한 줄로 만든다. 아래에서 노선·거리·금액에 그대로 다시 쓴다
# 역이 n 개면 구간은 n-1 개. 역 목록을 한 칸 밀어 zip 하면 구간마다 짝지어진다
def draw_path(names, labels):
    drawn = names[0]
    for name, label in zip(names[1:], labels):
        drawn += f' -({label})-> {name}'
    return drawn

for r in rows:
    print(draw_path(r['경로'], r['구간노선']))

> 같은 강남에 이르는 길이 정말 둘입니다. 지나는 역이 다르고, **두 번째 구간에서 타는 노선이 갈립니다**(첫 구간은 둘 다 같은 노선입니다). 가변길이 패턴은 **경로마다 한 행**을 돌려주므로 도착지가 같아도 길이 다르면 행이 따로 생깁니다.

### 🖐️ 함께 따라하기: 정확히 두 단계 건너간 계좌

이번에는 **송금 그래프**(`Account`·`TRANSFER`)에서 같은 기술을 써 봅니다. 데모가 쓴 전철 그래프가 아니라 위에서 함께 적재한 송금 그래프입니다.

전철과 결정적으로 다른 점이 하나 있습니다. **돈은 한쪽으로만 흐릅니다.** 그래서 화살표를 빼면 안 되고 `-[:TRANSFER*2..2]->` 처럼 **방향을 그대로 둡니다.** 화살표를 빼면 내가 보낸 돈과 나에게 들어온 돈이 한 덩어리로 섞입니다.

`A-1001` 계좌에서 **정확히 두 단계**(`*2..2`) 건너간 계좌를 찾으세요. 바로 다음 계좌인 `A-1002`(한 단계)가 결과에서 **빠지는지** 확인하는 게 핵심입니다.

이어서 같은 자리에서 `*1..3` 으로도 한 번 실행해, 범위를 넓히면 계좌가 **더 늘어나는지** 비교해 보세요.

**확인 기준**: `*2..2` 는 **2곳**(A-1003·A-1005), `*1..3` 은 **6곳** 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) Account 레이블과 TRANSFER 관계로, A-1001 에서 *2..2 건너간 계좌를 찾아 실행한다
#    화살표를 살린 -[:TRANSFER*2..2]-> 로 쓴다 (한 단계 계좌가 빠지는지 확인)
# 2) 같은 쿼리를 *1..3 으로 한 번 더 실행해 계좌가 늘어나는지 비교한다
# 3) 둘 다 RETURN DISTINCT b.name, ORDER BY 로 오름차순 정렬해 출력한다

위에서 쓴 `*1..2` 말고도 표에 적어 둔 표기가 여럿입니다. 그중 헷갈리기 쉬운 둘, **하한을 안 적은 `*..2`** 와 **하한이 0인 `*0..2`** 를 `*1..2` 와 나란히 돌려 봅니다. 표에 적힌 말이 실제로 그런지 눈으로 확인하는 자리입니다.

In [ ]:
# 하한 0 이 출발역 자신을 답에 넣는지 나란히 물어 본다
# *0..2 는 관계를 0번 타는 것도 허용해 출발역 자신이 들어온다
hop1 = run_cypher("MATCH (a:Station {name:'남부터미널'})-[:NEXT_TO*1..2]-(b:Station) "
                  "RETURN DISTINCT b.name AS 역 ORDER BY 역")
hop0 = run_cypher("MATCH (a:Station {name:'남부터미널'})-[:NEXT_TO*0..2]-(b:Station) "
                  "RETURN DISTINCT b.name AS 역 ORDER BY 역")
# *..2 도 함께 물어 정말 *1..2 와 같은지 본다
hop_short = run_cypher("MATCH (a:Station {name:'남부터미널'})-[:NEXT_TO*..2]-(b:Station) "
                       "RETURN DISTINCT b.name AS 역 ORDER BY 역")
print('*1..2 :', [r['역'] for r in hop1])
print('*..2  :', [r['역'] for r in hop_short])
print('*0..2 :', [r['역'] for r in hop0])

> `*0..2` 쪽이 한 개 많고, 그 하나가 **출발역 남부터미널 자신**입니다. 이름순으로 정렬했으므로 자기 자신은 맨 앞이 아니라 목록 **4번째**에 끼어 있습니다. 눈으로 찾지 말고 파이썬에서 `'남부터미널' in 목록` 으로 확인하는 편이 확실합니다.

> ⚠️ **상한을 열어 둘수록 훑는 경로가 늘어납니다.** 다만 무엇이 먼저 벽에 부딪히는지는 **무엇을 RETURN 하는가**에 달려 있습니다. 이 전철 그래프에서 실제로 재 보면 이렇습니다.

> - **도착역 이름만** 받으면(`RETURN DISTINCT b.name`) 서울역에서 `*1..4` 가 78곳, `*1..20` 이 527곳인데 둘 다 눈 깜짝할 새에 끝납니다. 도착역 단위로 잘라 내기 때문입니다.
> - 그런데 **경로 값을 함께 꺼내면**(`length(p)` 처럼) 경로를 전부 만들어야 합니다. 같은 서울역에서 `*1..8` 이 6,258행, `*1..10` 이 29,433행, `*1..12` 가 133,425행입니다. 상한을 2 늘릴 때마다 경로가 네 배쯤 늘어나고, 걸리는 시간도 그만큼 따라 늡니다.
> - **출발역을 안 박으면** 더 조용히 위험합니다. `MATCH (a:Station)-[:NEXT_TO*1..2]-(b:Station)` 은 눈 깜짝할 새에 끝나지만 **4,360행**이 돌아옵니다. 느려지지 않아서 실수를 눈치채기 어렵습니다.

> 정리하면 **가변길이 패턴에는 반드시 한쪽 끝에 이름을 박고**, 상한은 질문에 필요한 만큼만 여세요.

### 🖐️ 함께 따라하기: 출발 계좌와 그 아래 두 단계를 한 목록으로

**송금 그래프**에서 `A-1001` 계좌와, 거기서 **두 단계 이내**로 돈이 흘러간 계좌를 **한 목록**으로 뽑으세요. `A-1001` 자신이 목록에 **들어가야** 합니다. 자금 추적에서는 돈이 출발한 자리도 조사 대상이니까요.

**확인 기준**: **4곳**이 나오고 그 안에 `'A-1001'` 이 있습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) Account·TRANSFER 로 A-1001 에서 *0..2 범위의 계좌를 찾는다 (화살표를 살린다)
# 2) RETURN DISTINCT 로 이름을 받아 ORDER BY 로 정렬한다
# 3) 결과 목록과 'A-1001' 이 그 안에 있는지를 함께 출력한다

### ✅ 바로 확인 퀴즈

**1.** `-[:NEXT_TO*2..2]-` 는 무엇을 찾나요?

<details><summary>정답 보기</summary>

**정확히 두 칸** 떨어진 역만 찾습니다. 최소·최대가 둘 다 2라서 딱 두 정거장 거리인 역만 나옵니다.

</details>

**2.** 같은 역이 여러 번 나오지 않게 하려면 `RETURN` 에 무엇을 붙이나요? 왜 여러 번 나오나요?

<details><summary>정답 보기</summary>

**`DISTINCT`** 를 붙입니다(`RETURN DISTINCT b.name`). 그 역에 이르는 **경로가 여러 갈래**면 가변길이 패턴이 경로마다 한 행씩 돌려주기 때문에 같은 역이 여러 번 나옵니다.

</details>

**3.** `*..3` 은 몇 칸부터 몇 칸까지인가요?

<details><summary>정답 보기</summary>

**1 칸부터 3 칸까지** 입니다. 하한을 안 적으면 1 로 봅니다. 즉 `*..3` 은 `*1..3` 의 줄임입니다.

</details>

**4.** 출발 노드 자신을 결과에 넣으려면 어떻게 하나요?

<details><summary>정답 보기</summary>

하한을 **0** 으로 내립니다(`*0..2`). 관계를 한 번도 안 타는 길이 0 인 경로가 허용되므로 출발 노드가 결과에 들어옵니다.

</details>

---
# 2. 최단 경로

"닿을 수 있나"가 아니라 "**가장 적은 정거장으로 어떻게 가나**"를 묻는 자리입니다.

- **2-1** 최단 경로 한 갈래를 구합니다.
- **2-2** 가장 짧은 길이 여럿일 때 전부 받습니다.

## 2-1. `shortestPath` 로 가장 짧은 길 찾기

### 왜 필요할까요?
가능한 경로를 다 뒤져 제일 짧은 걸 고르는 일을 Cypher 가 **한 함수**로 해 줍니다. 앞 절에서 본 것처럼 상한 없는 `*` 를 그냥 열면 경로가 수만 갈래로 늘어나는데, `shortestPath` 는 짧은 쪽부터 찾아 **한 갈래를 찾으면 멈추기** 때문에 상한 없이 써도 됩니다.

### 문법: `shortestPath((...)-[:R*]-(...))`
경로 전체를 변수 **`p`** 에 담고, 관계에 **`*`**(길이 제한 없음)를 준 뒤 `shortestPath(...)` 로 감쌉니다.

```text
MATCH p = shortestPath( (출발)-[:NEXT_TO*]-(도착) )
```

- `p =` : 찾은 경로 전체를 변수에 담는 **경로 변수**
- `*` : 길이 제한을 두지 않음(몇 칸이 걸릴지 모르니까)
- `-[:NEXT_TO]-` : 방향 없이(전철은 양방향)

### 무엇을 물을 때 무엇을 쓰나
| 묻는 것 | 쓰는 문법 |
|---|---|
| 몇 칸 안에 무엇이 있는지 | 가변길이 `*1..3`(상한을 질문이 정한다) |
| 목적지까지 어떻게 가는지 | `shortestPath`(경로 자체가 답) |
| 목적지에 닿기는 하는지 | `shortestPath` 로 물어 행이 오는지 본다 |

<img src="images/shortestPath_경로.png" width="760">

*바로 아래 데모의 경로입니다. 5 정거장을 이동해 역 6개를 지나고, 종로3가 에서 한 번 갈아탑니다.*

In [ ]:
# 서울역에서 경복궁 까지 최단 경로. 경로를 p 에 담아 역 이름을 꺼낸다
# 몇 정거장인지 모르니 * 로 열어 두고 shortestPath 에 맡긴다
# [n IN nodes(p) | n.name] 은 노드 목록에서 이름만 뽑는다(3-2 에서 자세히)
rows = run_cypher("""
MATCH p = shortestPath( (a:Station {name:'서울역'})-[:NEXT_TO*]-(b:Station {name:'경복궁'}) )
RETURN [n IN nodes(p) | n.name] AS 경로,
       [r IN relationships(p) | r.line] AS 구간노선
""")
# shortestPath 는 최단 경로를 한 갈래만 돌려주므로 결과도 한 행이다
print('서울역 -> 경복궁 최단 경로:', rows[0]['경로'])
# 노선은 구간에 붙어 있다. 어디서 갈아탔는지가 여기 드러난다
print('구간 노선     :', rows[0]['구간노선'])

In [ ]:
# 1-1 에서 만든 draw_path 를 그대로 쓴다. 역 사이에 그 구간의 노선이 끼워진다
print('그래프 모양으로 :', draw_path(rows[0]['경로'], rows[0]['구간노선']))

> `nodes(p)` 는 경로가 지나는 노드들을 **순서대로** 담은 목록입니다. 서울역 → 시청 → 종각 → 종로3가 → 안국 → 경복궁 순서로 나옵니다. 구간 노선을 보면 1호선 을 타고 가다 **종로3가** 에서 3호선 으로 갈아탄 사실이 그대로 보입니다.

> 이 짝은 최단 경로가 **한 갈래뿐**입니다(2-2 에서 확인하는 법을 배웁니다). 같은 길이의 다른 길이 있으면 `shortestPath` 가 어느 쪽을 줄지 정해져 있지 않아 실행할 때마다 답이 달라질 수 있습니다.

> ⚠️ **역 이름을 잘못 적으면** 맞는 경로가 하나도 없어 `rows` 가 **빈 목록**이 되고, `rows[0]` 이 `IndexError: list index out of range` 를 냅니다. 이 오류를 만나면 오타부터 확인하세요. 결과가 없을 수도 있는 쿼리는 `if rows:` 로 먼저 확인하거나, 다음 시간에 배울 `OPTIONAL MATCH` 로 빈 결과 대신 `null` 을 받는 방법을 씁니다.

### 🖐️ 함께 따라하기: 방향을 지킬 때와 무시할 때

**송금 그래프**입니다. `A-1008` 에서 `A-1001` 까지 최단 경로를 **두 번** 구해 나란히 비교해 보세요. 두 쿼리는 **화살표 하나만** 다릅니다.

1. **방향을 지켜서** `-[:TRANSFER*]->` : "A-1008 이 보낸 돈이 A-1001 까지 흘러갔나"를 묻습니다.
2. **방향을 무시해서** `-[:TRANSFER*]-` : "둘이 거래로 이어져 있기는 한가"를 묻습니다.

1번은 **결과가 아예 없을 수 있습니다.** 그러니 `rows[0]` 을 바로 꺼내지 말고 `len(rows)` 부터 출력하세요(빈 목록에서 `rows[0]` 을 꺼내면 `IndexError` 가 납니다).

**확인 기준**: 1번은 **0갈래**, 2번은 경로가 있고 **4단계**입니다. 돈은 A-1001 쪽에서 A-1008 쪽으로 흘렀지 그 반대로 흐른 적이 없습니다. **화살표 하나가 질문의 뜻을 통째로 바꿉니다.**

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) shortestPath 로 A-1008 에서 A-1001 까지 화살표를 살려(-[:TRANSFER*]->) 구하고 len() 을 출력한다
# 2) 같은 쿼리에서 화살표만 빼고(-[:TRANSFER*]-) 다시 구해 len() 과 length(p) 를 출력한다
# 3) 방향을 무시한 경로의 계좌 목록을 ' - ' 로 이어 붙여 출력한다

### ✅ 바로 확인 퀴즈

**1.** `shortestPath` 안의 관계에 길이를 `*1..3` 처럼 제한하지 않고 그냥 `*` 로 두는 이유는?

<details><summary>정답 보기</summary>

두 역이 **몇 정거장 떨어져 있는지 미리 모르기** 때문입니다. 길이를 열어 두면 필요한 만큼 이어서 가장 짧은 경로를 찾습니다. `shortestPath` 는 짧은 쪽부터 찾아 한 갈래를 찾으면 멈추므로 상한 없이 써도 경로를 전부 만들지 않습니다.

</details>

**2.** 경로 전체를 담는 **경로 변수**는 어떻게 잡나요?

<details><summary>정답 보기</summary>

패턴 앞에 **`p =`** 를 붙입니다(`MATCH p = shortestPath(...)`). 그 뒤 `p` 로 경로를 다룹니다.

</details>

## 2-2. 최단 경로가 여럿일 때: `allShortestPaths`

### 왜 필요할까요?
**가장 짧은 길이 꼭 하나라는 법은 없습니다.** 같은 정거장 수로 갈 수 있는 길이 두 갈래일 수 있습니다. 그런데 `shortestPath` 는 그중 **한 갈래만** 돌려줍니다. 어느 쪽을 줄지는 우리가 정하지 못합니다. "가는 방법을 전부 보여 달라"는 질문에는 다른 함수를 써야 합니다.

### 문법: `allShortestPaths`
쓰는 자리와 모양은 `shortestPath` 와 똑같고, **가장 짧은 길이를 가진 경로를 전부** 돌려줍니다.

```text
MATCH p = shortestPath( (출발)-[:NEXT_TO*]-(도착) )      가장 짧은 길 한 갈래만
MATCH p = allShortestPaths( (출발)-[:NEXT_TO*]-(도착) )  가장 짧은 길이 여럿이면 전부
```

그런 자리가 **광화문 → 서울역** 입니다. 4 정거장짜리 길이 두 갈래 있습니다.

<img src="images/최단경로_동점.png" width="760">

*광화문에서 서울역까지 4 정거장짜리 길이 두 갈래입니다. 가운데가 서로 다르고 시청 에서 다시 만납니다.*

In [ ]:
# 같은 질문을 두 함수로 물어 돌아오는 행 수를 비교한다
# 두 쿼리는 함수 이름 하나만 다르다. 나머지는 글자까지 같다
one = run_cypher("""
MATCH p = shortestPath( (a:Station {name:'광화문'})-[:NEXT_TO*]-(b:Station {name:'서울역'}) )
RETURN [n IN nodes(p) | n.name] AS 경로
""")
every = run_cypher("""
MATCH p = allShortestPaths( (a:Station {name:'광화문'})-[:NEXT_TO*]-(b:Station {name:'서울역'}) )
RETURN [n IN nodes(p) | n.name] AS 경로
""")
print('shortestPath 갈래 수:', len(one))
print('allShortestPaths 갈래 수:', len(every))

In [ ]:
# 두 갈래가 실제로 어떤 길인지 펼쳐 본다. 지나는 역과 구간 노선을 함께 받는다
rows = run_cypher("""
MATCH p = allShortestPaths( (a:Station {name:'광화문'})-[:NEXT_TO*]-(b:Station {name:'서울역'}) )
RETURN [n IN nodes(p) | n.name] AS 경로,
       [r IN relationships(p) | r.line] AS 구간노선,
       length(p) AS 정거장수
""")
for r in rows:
    print(r['정거장수'], '정거장:', draw_path(r['경로'], r['구간노선']))

> 두 갈래 모두 **4 정거장**입니다. 가운데 지나는 역이 서로 다르고 (도착 한 정거장 앞 **시청** 에서 다시 만납니다), 한쪽은 `5호선·1호선·1호선·1호선` 을 타고 갈아타기가 **1번**, 다른 한쪽은 `5호선·5호선·2호선·1호선` 을 타고 **2번**입니다.

> **어느 쪽이 더 낫다고 데이터가 말해 주지 않습니다.** 정거장 수만 보면 완전히 같습니다. 환승 횟수를 기준으로 삼기로 **우리가 정해야** 비로소 한쪽을 고를 수 있습니다.

> 그래서 `shortestPath` 를 쓸 때는 늘 물어야 합니다. **"같은 길이인 길이 또 있지는 않은가?"** 있는데 하나만 받아 놓고 "이것이 유일한 최단 경로"라고 말하면 그건 틀린 설명이 됩니다.

### 거리로 재면 어떨까

정거장 수는 같았습니다. 그런데 **구간마다 길이가 다릅니다.** `NEXT_TO` 관계에는 `km` 속성이 붙어 있습니다(두 역 좌표 사이의 **직선거리**입니다. 실제 선로 길이는 이보다 조금 깁니다).

`relationships(p)` 가 준 목록에서 `km` 만 뽑아 파이썬에서 더해 보면, 같은 정거장 수인데도 **거리가 다르다**는 것이 드러납니다.

In [ ]:
# [r IN relationships(p) | r.km] 가 구간 길이 목록이다
rows = run_cypher("""
MATCH p = allShortestPaths( (a:Station {name:'광화문'})-[:NEXT_TO*]-(b:Station {name:'서울역'}) )
RETURN [n IN nodes(p) | n.name] AS 경로,
       [r IN relationships(p) | r.km] AS 구간거리
""")
# 합계는 파이썬에서 낸다. Cypher 집계는 다음 시간에 배운다
# draw_path 에 노선 대신 구간 거리를 넘기면 어느 구간이 긴지 그 자리에서 보인다
for r in rows:
    total_km = round(sum(r['구간거리']), 2)
    # 소수 자릿수를 맞춰야 구간끼리 눈으로 견주기 쉽다
    km_labels = [f'{km:.2f}' for km in r['구간거리']]
    print(total_km, 'km :', draw_path(r['경로'], km_labels))

> 짧은 쪽이 **4.06 km**, 긴 쪽이 **4.41 km** 입니다. 차이는 0.35 km 밖에 안 되지만 **분명히 다릅니다.**

> 이제 기준이 둘입니다. **환승이 적은 길**과 **거리가 짧은 길**이 서로 다를 수 있습니다. 그래서 "최단" 이라는 말은 그 자체로는 뜻이 정해지지 않습니다. 무엇을 재는지 말해야 합니다.

> ⚠️ **`shortestPath` 는 거리를 볼 줄 모릅니다.** 정거장 수(홉 수)만 셉니다. 그래서 위처럼 같은 홉 수인 길을 **전부 받아 놓고 우리가 재서** 골라야 합니다. 거리를 직접 최소화하는 알고리즘(다익스트라)은 뒤 단원에서 GDS 라이브러리로 다룹니다.

### 🖐️ 함께 따라하기: 송금에도 두 갈래가 있다

**송금 그래프**에서 `A-1002` 에서 `A-1006` 로 가는 길을 두 함수로 각각 물어 보세요. 이번에는 **방향을 지킵니다**(`-[:TRANSFER*]->`).

1. `shortestPath` 로 구해 갈래 수를 출력합니다.
2. `allShortestPaths` 로 구해 갈래 수와 각 경로를 출력합니다. 경로마다 `relationships(p)` 로 **구간 금액**도 함께 받으세요.

**확인 기준**: 1번은 **1갈래**, 2번은 **2갈래**이고 둘 다 **2단계**입니다. 한쪽은 A-1003 경유, 다른 쪽은 A-1005 경유입니다. 자금이 **두 갈래로 쪼개져 나갔다가 같은 계좌에서 다시 만난** 모양입니다. 한 갈래만 보고 "이 돈은 저쪽을 거쳤다"고 말하면 나머지 절반을 놓칩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) shortestPath 로 A-1002 에서 A-1006 까지 화살표를 살려 구하고 갈래 수를 센다
# 2) 같은 쿼리에서 함수 이름만 allShortestPaths 로 바꿔 다시 구한다
# 3) 경로와 [r IN relationships(p) | r.amount] 로 구간 금액을 함께 받는다
# 4) 1-1 에서 만든 draw_path 에 금액을 넘겨 갈래마다 한 줄씩 출력한다

### ✅ 바로 확인 퀴즈

**1.** `shortestPath` 와 `allShortestPaths` 는 무엇이 다른가요?

<details><summary>정답 보기</summary>

둘 다 **가장 짧은 길이**의 경로를 찾지만, `shortestPath` 는 그중 **한 갈래만** 돌려주고 `allShortestPaths` 는 그 길이를 가진 경로를 **전부** 돌려줍니다.

</details>

**2.** "A 에서 B 로 가는 최단 경로는 이것입니다" 라고 말하기 전에 무엇을 확인해야 하나요?

<details><summary>정답 보기</summary>

**같은 길이의 다른 경로가 있는지**를 확인해야 합니다. `allShortestPaths` 로 세어 보고 한 갈래뿐일 때만 "이것이 최단 경로"라고 말할 수 있습니다. 여럿이면 그중 무엇을 고를지 **기준을 따로 정해야** 합니다.

</details>

---
# 3. 경로를 해부하고 조건 걸기

경로를 찾았으면 그 안에서 값을 꺼내 써야 합니다.

- **3-1** 경로에서 꺼내는 함수 셋을 익힙니다.
- **3-2** 그 목록에서 값만 뽑는 표현을 제대로 읽습니다.
- **3-3** 그 값으로 줄을 세웁니다.
- **3-4** 경로 위 구간에 조건을 겁니다.

## 3-1. `length`·`nodes`·`relationships` 로 꺼내기

### 왜 필요할까요?
"**몇 정거장이었나**", "**어떤 역을 지났나**", "**어느 노선을 탔나**"를 꺼내 써야 합니다. 경로 변수 `p` 에서 이 정보를 뽑는 함수가 셋 있습니다.

### 문법: 경로에서 꺼내는 세 가지
| 함수 | 무엇을 주나 | 전철에서 |
|---|---|---|
| `length(p)` | 경로의 **관계(엣지) 개수** | 몇 **정거장**을 이동했나 |
| `nodes(p)` | 경로가 지난 **노드 목록**(순서대로) | 어떤 **역들**을 지났나 |
| `relationships(p)` | 경로가 탄 **관계 목록**(순서대로) | 구간마다 어느 **노선**이었나 |

정거장 수(`length`)는 **역 개수보다 하나 적습니다**. 역이 6개면 그 사이 이동은 5번이니까요. `relationships(p)` 는 그 이동을 순서대로 주므로, 관계에 붙은 속성(`line`)을 꺼내면 어디서 갈아탔는지가 드러납니다.

<img src="images/length_nodes.png" width="760">

*서울역 - 시청 - 을지로입구 - 을지로3가 - 을지로4가 - 동대문역사문화공원 으로 이어지는 길을 예로 든 그림입니다. 역 6개를 잇는 화살표는 5개라 `nodes(p)` 는 6, `length(p)` 는 5입니다.*

In [ ]:
# 서울역에서 이태원 까지 최단 경로를 해부한다. 이동 수·지나는 역·구간 노선
# length(p) 는 탄 관계 개수(= 이동 수), nodes(p) 는 지나는 노드 목록
# 노선은 구간에 붙은 line 속성이라 relationships(p) 에서 꺼낸다
rows = run_cypher("""
MATCH p = shortestPath( (a:Station {name:'서울역'})-[:NEXT_TO*]-(b:Station {name:'이태원'}) )
RETURN length(p) AS 정거장수,
       [n IN nodes(p) | n.name] AS 지나는역,
       [r IN relationships(p) | r.line] AS 구간노선
""")
print('정거장 수:', rows[0]['정거장수'])
print('지나는 역:', rows[0]['지나는역'])
print('구간 노선:', rows[0]['구간노선'])

In [ ]:
# 역과 구간 노선을 번갈아 끼우면 한 줄로 읽힌다
# zip 은 앞에서부터 짝짓는다. 역이 하나 더 많아 마지막 역은 따로 붙인다
chain = ''.join(f"{stop} -({line})-> " for stop, line in zip(rows[0]['지나는역'], rows[0]['구간노선']))
print('그래프 모양으로 :', chain + rows[0]['지나는역'][-1])   

> 4 정거장을 이동해 5개 역을 지납니다. 구간 노선을 보면 앞 2구간이 4호선, 나머지가 6호선 입니다. 즉 **삼각지 에서 갈아탔다**는 사실이 노선 목록에 그대로 드러납니다. 역 이름 목록(`nodes`)만 봐서는 알 수 없고, 구간 목록(`relationships`)을 봐야 보입니다.

### 🖐️ 함께 따라하기: 계좌 개수와 송금 횟수의 차이 확인

송금 그래프에서 `A-1001` 에서 `A-1009` 까지 최단 경로를 **방향을 지켜** 구해, **거친 계좌 목록**과 **송금 횟수**(`length`), 그리고 `relationships(p)` 로 **구간 금액과 날짜**를 함께 출력하세요. 그리고 파이썬에서 계좌 목록의 길이(`len`)를 세어, **송금 횟수가 계좌 개수보다 정확히 하나 적은지** 직접 확인합니다.

**확인 기준**: 계좌 **4곳**, 송금 횟수 **3** 입니다. 금액이 `8,000,000 → 2,500,000 → 600,000` 으로 **갈수록 작아지는데**, 큰 돈을 잘게 쪼개 흘린 자국입니다. 금액과 날짜는 계좌가 아니라 **구간에 붙어 있어** `nodes(p)` 로는 못 꺼냅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) shortestPath 로 A-1001 에서 A-1009 까지 화살표를 살려 경로 p 를 찾는다
# 2) length(p) 와 [n IN nodes(p) | n.name], 구간 금액·날짜를 함께 RETURN 한다
# 3) 계좌 목록의 len 과 송금 횟수를 나란히 출력해 차이가 1인지 확인한다

### ✅ 바로 확인 퀴즈

**1.** 경로가 역 4개를 지난다면 `length(p)` 값은 얼마인가요?

<details><summary>정답 보기</summary>

**3** 입니다. `length` 는 관계(이동) 개수라, 역 개수보다 하나 적습니다.

</details>

**2.** 경로가 **어느 노선을 탔는지**를 알려면 어떤 함수를 쓰나요?

<details><summary>정답 보기</summary>

**`relationships(p)`** 로 관계 목록을 얻고, `[r IN relationships(p) | r.line]` 로 관계 속성을 뽑습니다. 역 이름이 필요하면 `nodes(p)`, 이동 수만 필요하면 `length(p)` 입니다.

</details>

## 3-2. 목록에서 값만 뽑기: 리스트 표현식

### 왜 필요할까요?
앞 절에서 이미 `[n IN nodes(p) | n.name]` 을 썼습니다. 이제 그 대괄호 안이 무슨 뜻인지 제대로 봅니다.

`nodes(p)` 가 돌려주는 것은 **역 이름 목록이 아니라 노드 자체의 목록**입니다. 노드는 이름 말고도 자기가 가진 속성을 통째로 달고 오기 때문에, 뽑지 않고 그대로 찍으면 이렇게 나옵니다.

```text
[{'name': '서울역', 'lines': ['1호선', '4호선', 'GTX-A', '경의·중앙선', '공항철도']}, {'name': '숙대입구', 'lines': ['4호선']}, ...]
```

우리가 읽고 싶은 것은 `서울역 → 숙대입구 → 삼각지 ...` 인데, 덩어리 속에서 이름을 눈으로 찾아내야 합니다. 지나는 역이 스무 곳이면 한 줄이 화면을 넘어갑니다. 그래서 **목록을 훑어 필요한 값만 뽑아내는 문법**을 씁니다. 파이썬의 리스트 컴프리헨션이 하는 일과 똑같습니다.

### 문법: `[변수 IN 목록 | 표현식]`

```text
[ n IN nodes(p) | n.name ]
  │    │          └ 표현식: 원소마다 꺼낼 값
  │    └ 목록: 하나씩 훑어볼 대상
  └ 변수: 원소 하나를 담아 둘 이름
```

목록의 원소를 하나씩 `n` 에 담고, **세로선(`|`) 뒤 표현식**을 계산한 값을 모아 새 목록으로 돌려줍니다. 원소를 5개 넣으면 5개가 나옵니다. 돌아오는 것은 값만 든 평범한 목록이라, 파이썬에서 `' → '.join(...)` 으로 바로 이어 붙일 수 있습니다.

세로선 뒤는 **값 하나를 만드는 식**이면 무엇이든 됩니다. 그래서 속성 하나만 꺼낼 수 있는 것이 아니라, 여러 값을 **리스트**(`[ ]`)나 **딕셔너리**(`{ }`)로 묶어 한 번에 꺼낼 수도 있습니다.

```text
[ r IN relationships(p) | [r.line, r.km] ]              값 둘을 리스트로 묶는다
[ r IN relationships(p) | {노선: r.line, 거리: r.km} ]    딕셔너리로 묶어 이름을 붙인다
```

앞엣것은 리스트의 리스트(`[[...], [...]]`), 뒤엣것은 딕셔너리의 리스트(`[{...}, {...}]`)로 돌아옵니다. **딕셔너리 쪽이 이름이 붙어 나중에 읽기 쉽습니다.** 계산식(`r.km * 1000`)이나 문자열 조립도 같은 자리에 그대로 적을 수 있습니다.

**거르면서 뽑을 때**는 세로선 앞에 `WHERE` 를 붙입니다.

```text
[ r IN relationships(p) WHERE r.line = '6호선' | r.line ]
                        └ 거르는 조건: 이 조건을 통과한 원소만 남긴다
```

조건을 통과한 원소만 표현식을 계산하므로 결과가 **원래보다 짧아질 수 있습니다**.

`WHERE` 는 **노드 목록에도 똑같이** 걸립니다. 아래는 경로가 지나는 역 가운데 6호선이 서는 역만 남깁니다. 거르는 조건에 쓰는 것은 그 원소가 **자기가 가진 속성**입니다.

```text
[ n IN nodes(p) WHERE '6호선' IN n.lines | n.name ]
```

### 파이썬과 나란히 놓고 보기

| 하는 일 | 파이썬 | Cypher |
|---|---|---|
| 원소마다 값 하나 뽑기 | `[n.name for n in nodes]` | `[n IN nodes(p) \| n.name]` |
| 조건에 맞는 것만 뽑기 | `[r.line for r in rels if r.line == '6호선']` | `[r IN relationships(p) WHERE r.line = '6호선' \| r.line]` |
| 꺼낼 값을 적는 자리 | 맨 **앞** | 세로선 뒤, 맨 **뒤** |

바꿔 읽을 것은 셋뿐입니다. `for n in` 세 낱말이 `n IN` 두 낱말로 줄고, `if` 자리에 `WHERE` 가 오고, **꺼낼 값을 앞이 아니라 세로선 뒤에** 적습니다. 같음 비교가 `==` 가 아니라 **`=`** 하나인 것도 파이썬과 다릅니다.

In [ ]:
# 3-1 에서 해부한 그 경로(서울역 -> 이태원)를 여섯 가지로 찍어 견준다
# 1) nodes(p) 를 그대로 두면 노드가 통째로 돌아온다
# 2) [n IN nodes(p) | n.name] 은 이름만 새 목록으로 만든다
# 3) 관계 목록도 문법이 같다. name 대신 line 을 꺼낼 뿐이다
# 4) 세로선 앞에 WHERE 를 붙이면 통과한 원소만 남는다
# 5) 세로선 뒤에 딕셔너리를 적으면 한 구간에서 값을 둘 이상 함께 꺼낸다
# 6) WHERE 는 노드 목록에도 똑같이 걸린다. 노드가 자기 속성으로 걸러진다
rows = run_cypher("""
MATCH p = shortestPath( (a:Station {name:'서울역'})-[:NEXT_TO*]-(b:Station {name:'이태원'}) )
RETURN nodes(p) AS 노드통째로,
       [n IN nodes(p) | n.name] AS 역이름만,
       [r IN relationships(p) | r.line] AS 구간노선,
       [r IN relationships(p) WHERE r.line = '6호선' | r.line] AS 고른구간,
       [r IN relationships(p) | {노선: r.line, 거리: r.km}] AS 노선과거리,
       [n IN nodes(p) WHERE '6호선' IN n.lines | n.name] AS 그노선역만
""")
print('1) nodes(p) 그대로 :', rows[0]['노드통째로'])
print('2) 이름만 뽑기     :', rows[0]['역이름만'])
print('3) 구간 노선       :', rows[0]['구간노선'])
print('4) 6호선 구간만    :', rows[0]['고른구간'])
print('5) 노선과 거리 함께 :', rows[0]['노선과거리'])
print('6) 6호선 서는 역만  :', rows[0]['그노선역만'])

### 🖐️ 함께 따라하기: 큰 금액 구간만 골라 뽑기

**송금 그래프**에서 `A-1004` 에서 `A-1008` 까지 최단 경로를 **방향을 지켜** 구하고, 리스트 표현식 **세 개**를 한 번에 받아 출력하세요.

1. 거친 계좌 이름만
2. 구간 금액 전부
3. 그중 **1,000,000원 이상**인 금액만(세로선 앞에 `WHERE r.amount >= 1000000` 를 붙입니다)

**확인 기준**: 계좌 **4곳**, 금액 **3개**, 1,000,000원 이상은 **2개**입니다. 3번 목록만 짧아지는 것이 핵심입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) shortestPath 로 A-1004 에서 A-1008 까지 화살표를 살려 경로 p 를 찾는다
# 2) [n IN nodes(p) | n.name] 로 계좌 이름을, [r IN relationships(p) | r.amount] 로 금액을 받는다
# 3) 세로선 앞에 WHERE r.amount >= 1000000 를 붙인 목록도 함께 받아 셋을 나란히 출력한다

### ✅ 바로 확인 퀴즈

**1.** `[n IN nodes(p) | n.name]` 에서 세로선 뒤를 `n` 으로 바꾸면 결과가 어떻게 되나요?

<details><summary>정답 보기</summary>

`nodes(p)` 를 그대로 찍은 것과 **같아집니다**. 오류는 나지 않고, 속성이 다 붙은 노드 덩어리 목록이 그대로 돌아옵니다. 세로선 뒤에는 꺼낼 값을 적어야 합니다.

</details>

**2.** 파이썬 `[r.line for r in rels if r.line == '6호선']` 을 Cypher 리스트 표현식으로 옮기면?

<details><summary>정답 보기</summary>

`[r IN relationships(p) WHERE r.line = '6호선' | r.line]` 입니다. `for r in` 은 `r IN`, `if` 는 `WHERE` 로 바뀌고, 꺼낼 값 `r.line` 은 맨 뒤로 갑니다. Cypher 의 같음 비교는 `==` 가 아니라 **`=`** 하나입니다.

</details>

## 3-3. 경로 길이로 줄 세우기

### 왜 필요할까요?
`length(p)` 는 출력만 하라고 있는 값이 아닙니다. **정렬 기준**으로 쓰면 "이 역에서 가장 먼 역은 어디인가", "가까운 순서로 보여 달라"에 바로 답할 수 있습니다.

### 문법: `ORDER BY length(p)`
`RETURN` 에 적은 별칭으로도, `length(p)` 표현 그대로도 정렬할 수 있습니다.

```text
MATCH p = shortestPath( (a)-[:NEXT_TO*]-(b) )
RETURN b.name AS 역, length(p) AS 정거장수
ORDER BY 정거장수 DESC, 역        ← 먼 순서로, 동점이면 이름순
LIMIT 13                          ← 위에서 13 줄만
```

도착 노드를 하나로 못 박지 않으면 **닿을 수 있는 모든 역까지의 최단 거리**가 한 번에 나옵니다. 이 그래프에서는 그게 **655줄**이라 화면에 다 담기지 않으므로 지난 시간에 배운 `LIMIT` 으로 위쪽만 받습니다. 그리고 출발역 자신은 `WHERE` 로 빼야 합니다. `shortestPath` 는 출발과 도착이 같으면 오류를 내기 때문입니다.

In [ ]:
# 서울역에서 각 역까지 최단 몇 정거장인지 재고 먼 순서로 줄 세운다
# 도착지(b)에 이름을 안 박으면 닿을 수 있는 모든 역이 도착 후보가 된다
# 출발역 자신은 빼야 한다. shortestPath 는 출발과 도착이 같은 행에서 오류를 낸다
# 정거장 수 내림차순, 같으면 이름순. 순서를 유일하게 만든다
rows = run_cypher("""
MATCH p = shortestPath( (a:Station {name:'서울역'})-[:NEXT_TO*]-(b:Station) )
WHERE b.name <> '서울역'
RETURN b.name AS 역, length(p) AS 정거장수
ORDER BY 정거장수 DESC, 역
LIMIT 13
""")
for r in rows:
    print(r['정거장수'], '정거장:', r['역'])

> ⚠️ **`length(p)` 는 `ORDER BY` 에는 써도 되지만 `WHERE` 에 걸면 뜻이 달라집니다.**

> `shortestPath(...)` 뒤에 `WHERE length(p) >= 4` 를 붙이면 Cypher 는 이렇게 읽지 **않습니다**.
> *"가장 짧은 길을 하나 구해 놓고, 그게 4칸 이상인지 본다"*

> 대신 이렇게 읽습니다.
> *"**4칸 이상이면서 가장 짧은** 길을 찾아라"*

> 그래서 진짜 최단 경로가 2칸이면 그것을 버리고 3칸짜리를, 그것도 조건에 안 맞으면 4칸짜리를 찾는 식으로 **조건에 맞는 것이 나올 때까지 경로를 계속 만들어 봅니다.** `shortestPath` 의 장점인 "하나 찾으면 멈춘다"가 사라지는 것입니다. 이 659역 그래프에서는 그 탐색이 1분이 넘도록 끝나지 않아 **커널이 멈춘 것처럼 보입니다**(직접 실행하지 마세요).

> **구해 온 결과를 거르고 싶은 것뿐이라면** 다음 시간에 배울 `WITH` 로 한 단계 넘긴 뒤에 `WHERE` 를 걸어야 합니다. 지금은 받아 온 목록을 파이썬에서 걸러도 됩니다.

`LIMIT` 을 떼면 몇 줄이 오는지, 그리고 역이 659곳인데 왜 그만큼 오지 않는지 아래 셀에서 확인합니다. 결과를 화면에 다 찍지 않고 **개수만** 세는 것이 요령입니다.

In [ ]:
# LIMIT 을 뗀 같은 쿼리. 결과를 그대로 print 하면 화면이 덮이므로 개수만 센다
rows_all = run_cypher("""
MATCH p = shortestPath( (a:Station {name:'서울역'})-[:NEXT_TO*]-(b:Station) )
WHERE b.name <> '서울역'
RETURN b.name AS 역, length(p) AS 정거장수
ORDER BY 정거장수 DESC, 역
""")
print('LIMIT 없이 받은 행 수:', len(rows_all))

In [ ]:
# 왜 행이 역 수보다 적을까. 빠진 역을 집합 빼기로 찾는다
reached = {r['역'] for r in rows_all}
all_stations = {r['역'] for r in run_cypher("MATCH (n:Station) RETURN n.name AS 역")}
print('서울역에서 닿지 않는 역:', sorted(all_stations - reached - {'서울역'}))

> 역은 659곳인데 행은 655줄입니다. 출발역 서울역을 뺐고, **제1터미널·제2터미널·탑승동** 세 곳은 나머지 노선과 이어져 있지 않아 최단 경로 자체가 없습니다. 닿지 않는 노드는 **거리가 0 이나 `null` 로 나오는 게 아니라 행 자체가 생기지 않습니다.** "없는 것도 빠뜨리지 않고 세는" 방법은 다음 시간의 `OPTIONAL MATCH` 에서 배웁니다.

### 🖐️ 함께 따라하기: 출발 계좌에서 가장 멀리까지 흘러간 계좌

**송금 그래프**에서 `A-1001` 기준으로 각 계좌까지 **방향을 지켜** 최단 몇 단계인지 재고, **먼 순서**로 출력하세요. 동점을 대비해 **보조 정렬키로 계좌 이름**을 주고, `A-1001` 자신은 `WHERE` 로 빼세요(빼지 않으면 오류가 납니다). 계좌는 몇 개 안 되니 `LIMIT` 은 걸지 않습니다.

출력이 끝나면 **결과에 없는 계좌가 무엇인지** 찾아, 왜 빠졌는지 설명할 수 있어야 합니다.

**확인 기준**: **9줄**이 나오고 맨 위는 **A-1008(5단계)** 입니다. 계좌는 11개인데 9줄뿐입니다. 출발 계좌 `A-1001` 을 뺐고, **`A-1011` 은 송금 기록이 하나도 없어 애초에 닿지 않기 때문**입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) shortestPath 로 A-1001 에서 모든 Account 까지 경로 p 를 찾는다 (도착지에 이름을 박지 않는다)
# 2) WHERE 로 A-1001 자신을 뺀다
# 3) 이름과 length(p) 를 RETURN 하고 단계수 내림차순, 이름 오름차순으로 정렬해 출력한다
# 4) 전체 계좌 이름 집합에서 닿은 계좌와 출발 계좌를 빼서 빠진 계좌를 찾는다

### ✅ 바로 확인 퀴즈

**1.** 도착 노드에 이름을 적지 않고 `shortestPath` 를 쓰면 무엇이 나오나요?

<details><summary>정답 보기</summary>

출발 노드에서 **닿을 수 있는 모든 노드까지의 최단 경로**가 한 행씩 나옵니다. 그래서 `length(p)` 를 함께 받으면 **거리 표**가 됩니다. 닿지 못하는 노드는 행이 아예 생기지 않습니다.

</details>

**2.** `WHERE b.name <> '서울역'` 을 빼면 어떻게 되나요?

<details><summary>정답 보기</summary>

출발과 도착이 **같은 행**이 생겨 오류가 납니다. `shortestPath` 는 시작과 끝이 같은 경로를 찾지 못합니다.

</details>

## 3-4. 경로 위 구간에 조건 걸기: `all`·`any`·`none`

### 왜 필요할까요?
지금까지는 경로를 찾은 **뒤에** 내용을 꺼내 봤습니다. 그런데 실제 질문은 대개 경로 **자체에 조건**이 붙습니다. "**환승 없이** 4호선만 타고 갈 수 있는 역", "**모든 구간이** 60분 이하인 배송 경로", "1호선을 **한 번도 안 타는** 길". 이건 경로가 지나는 구간을 **전부 훑어 봐야** 답할 수 있습니다.

### 문법: 목록의 원소를 전부 검사하는 세 함수
`relationships(p)`·`nodes(p)` 가 주는 것은 **목록**입니다. 그 목록의 원소가 조건을 만족하는지를 세 가지 방식으로 물을 수 있습니다.

| 함수 | 참이 되는 때 | 우리 말로 |
|---|---|---|
| `all(r IN 목록 WHERE 조건)` | **모든** 원소가 조건을 만족 | 전부 그런가 |
| `any(r IN 목록 WHERE 조건)` | **하나라도** 만족 | 하나라도 있나 |
| `none(r IN 목록 WHERE 조건)` | **하나도** 만족하지 않음 | 하나도 없나 |

```text
all(r IN relationships(p) WHERE r.line = '4호선')
    └ 하나씩 꺼내 담을 이름   └ 검사할 목록      └ 각 원소가 만족해야 할 조건
```

3-2 의 리스트 표현식과 앞부분이 똑같이 생겼습니다. 다른 점은 **값을 뽑아 목록을 만드는 게 아니라 참·거짓 하나를 돌려준다**는 것입니다.

`RETURN` 에 적으면 참·거짓을 **값으로** 받고, `WHERE` 에 적으면 그 조건을 만족하는 **경로만 남깁니다.**

<img src="images/경로위_조건.png" width="760">

*3-1 에서 해부한 그 경로(서울역 → 이태원)를 세 가지로 묻습니다. 모든 구간이 한 노선인가(`all`), 다른 노선이 하나라도 있나(`any`), 어떤 노선을 한 번도 안 탔나(`none`).*

In [ ]:
# 세 경로를 같은 세 질문으로 물어 참·거짓을 값으로 받는다
# relationships(p) 의 원소 r 마다 r.line 을 본다
def check_lines(dest):
    """서울역에서 dest 역까지 최단 경로의 구간 노선을 세 가지로 검사해 dict 로 돌려준다."""
    rows = run_cypher("""
    MATCH p = shortestPath( (a:Station {name:'서울역'})-[:NEXT_TO*]-(b:Station {name:$dest}) )
    RETURN all(r IN relationships(p) WHERE r.line = '4호선') AS 전부_4호선,
           any(r IN relationships(p) WHERE r.line = '6호선') AS 포함_6호선,
           none(r IN relationships(p) WHERE r.line = '1호선') AS 안탐_1호선
    """, dest=dest)
    return rows[0]

print('서울역 -> 동대문역사문화공원 :', check_lines('동대문역사문화공원'))
print('서울역 -> 이태원 :', check_lines('이태원'))
print('서울역 -> 종로5가 :', check_lines('종로5가'))

> 세 경로의 구간 노선은 이렇습니다.

| 도착역 | 구간 노선 | `all`(전부 4호선) | `any`(포함 6호선) | `none`(안 탐 1호선) |
|---|---|---|---|---|
| 동대문역사문화공원 | `4호선·4호선·4호선·4호선` | True | False | True |
| 이태원 | `4호선·4호선·6호선·6호선` | False | True | True |
| 종로5가 | `1호선·1호선·1호선·1호선` | False | False | False |

> 동대문역사문화공원 까지는 **전 구간이 4호선** 이라 `all` 이 참입니다. 이태원 까지는 삼각지 에서 6호선 으로 갈아타므로 `all` 이 거짓이 되고 `any` 가 참이 됩니다. 종로5가 까지는 전 구간이 1호선 이라 `none` 마저 거짓입니다. **세 함수의 참·거짓이 빈칸 없이 채워지는지** 표로 확인하세요.

In [ ]:
# 이번에는 조건을 WHERE 에 걸어 '그런 경로만' 남긴다
# shortestPath 없이 가변길이로 열어 두고, 모든 구간이 한 노선인 경로만 통과시킨다
# 상한 4 는 화면에 담기는 크기. 경로 조건을 거는 쿼리는 상한을 꼭 두자
rows = run_cypher("""
MATCH p = (a:Station {name:'서울역'})-[:NEXT_TO*1..4]-(b:Station)
WHERE all(r IN relationships(p) WHERE r.line = '4호선')
RETURN DISTINCT b.name AS 역
ORDER BY 역
""")
print('서울역에서 환승 없이 4호선만 타고 4칸 안에 닿는 역:', [r['역'] for r in rows])   

In [ ]:
# 조건이 얼마나 걸러 냈는지 보려고 조건 없이 다시 센다
free = run_cypher("""
MATCH (a:Station {name:'서울역'})-[:NEXT_TO*1..4]-(b:Station)
RETURN DISTINCT b.name AS 역
""")
print('조건 없이 4칸 안:', len(free), '곳')

### 🖐️ 함께 따라하기: 큰 돈만 따라가면 제자리로 돌아온다

**송금 그래프**에서 `A-1001` 에서 출발해 **전 구간이 4,000,000원 이상**인 송금만 타고 닿을 수 있는 계좌를 찾으세요. 가변길이 상한 5(`*1..5`)으로 **방향을 지켜** 열고 `all` 을 `WHERE` 에 겁니다(구간 속성 이름은 `line` 이 아니라 `amount` 입니다).

결과 목록에 **`A-1001` 자신이 들어 있는지** 꼭 확인하세요. 들어 있다면 그건 큰 돈이 여러 계좌를 돌아 **출발한 자리로 되돌아왔다**는 뜻입니다.

**확인 기준**: **4곳**(A-1001·A-1002·A-1003·A-1004)이 나오고 그 안에 출발 계좌 `A-1001` 이 있습니다. 4,000,000원 미만 구간을 타야 닿는 계좌는 전부 빠집니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) A-1001 에서 출발하는 가변길이 경로 p 를 *1..5 로 연다 (화살표를 살린다)
# 2) WHERE 에 all(r IN relationships(p) WHERE r.amount >= 4000000) 을 건다
# 3) RETURN DISTINCT 로 계좌 이름을 받아 정렬해 출력하고, 출발 계좌가 그 안에 있는지 확인한다
# 4) 도착지를 출발 계좌로 못 박아 되돌아온 경로 자체를 펼쳐 본다

> **이 고리가 실무에서 뜻하는 것.** 돈이 여러 계좌를 거쳐 **출발한 자리로 되돌아오는 것**을 **순환거래**라고 합니다. 자금 세탁에서 돈의 출처를 흐리려고 일부러 만드는 모양이고, 매출을 부풀리는 가공거래에서도 같은 모양이 나옵니다. 그래서 이상거래 탐지에서 **고리 찾기는 기본 점검 항목**입니다.

> ⚠️ 다만 **고리가 있다고 곧 사기는 아닙니다.** 정산이나 환불처럼 정상적으로 되돌아오는 거래도 있습니다. 고리는 "여기를 들여다보라"는 신호이지 결론이 아닙니다. 그래서 실무에서는 고리에 **금액 하한**(여기서는 4,000,000원)이나 기간·계좌 수 같은 조건을 덧붙여 후보를 좁힙니다. 오늘 `all` 을 `WHERE` 에 건 것이 바로 그 금액 하한 조건입니다.

> 이 그래프는 계좌가 11개라 그림으로도 보입니다. 하지만 실제 거래 데이터는 계좌가 수백만 개입니다. 눈으로 못 찾는 것을 `*1..5` 한 줄로 찾아내는 것이 그래프 데이터베이스를 쓰는 이유입니다.

### ✅ 바로 확인 퀴즈

**1.** `all` 과 `any` 는 어떻게 다른가요?

<details><summary>정답 보기</summary>

`all` 은 목록의 **모든** 원소가 조건을 만족해야 참이고, `any` 는 **하나라도** 만족하면 참입니다. "전 구간이 4호선인가"는 `all`, "6호선 구간이 섞여 있나"는 `any` 입니다.

</details>

**2.** "1호선을 한 번도 타지 않는 경로"는 어떻게 쓰나요? `all` 로도 쓸 수 있나요?

<details><summary>정답 보기</summary>

`none(r IN relationships(p) WHERE r.line = '1호선')` 입니다. `all(r IN relationships(p) WHERE r.line <> '1호선')` 으로도 같은 뜻이 됩니다. `none(조건)` 은 `all(반대 조건)` 과 같습니다. 읽기 좋은 쪽을 고르면 됩니다.

</details>

---
## 🚀 응용 클론코딩: 경로 안내 함수

출발역과 도착역 이름을 받아, 최단 경로를 "**A → B → C (N 정거장, 환승 없음/있음)**" 형태의 한 문장으로 돌려주는 파이썬 함수 `guide(start, end)` 를 완성하세요. 안에서 `run_cypher` 로 `shortestPath` 를 구하고, `nodes(p)` 로 역 목록을, `length(p)` 로 정거장 수를 받고, **구간 노선이 한 종류인지**를 파이썬에서 `set` 으로 판단해 환승 여부를 붙입니다.

**예시**: `guide('여의도', '합정')` → `여의도 → 국회의사당 → 당산 → 합정 (3 정거장, 환승 있음)`

이어서 `guide('여의도', '공덕')` 도 불러 보세요. 이쪽은 전 구간이 5호선 이라 **환승 없음**이 나와야 합니다.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) guide(start, end) 정의
# 2) run_cypher 로 shortestPath 를 구하고 length(p)·nodes(p)·구간 노선을 RETURN (파라미터 $start,$end 사용)
# 3) 역 이름 목록을 예시와 같은 화살표 문자열로 잇고 '(N 정거장, 환승 없음/있음)' 을 붙여 return
# 4) guide('여의도', '합정') 와 guide('여의도', '공덕') 을 호출해 출력

---
## 참고: 같은 일을 하는 새 문법

공식 문서를 열면 오늘 배운 것과 **모양이 다른 문법**이 먼저 나옵니다. 최근 Cypher 는 그래프 질의 국제 표준(GQL)에 맞춘 표기를 도입했고, 문서는 그쪽을 본문으로 씁니다. 오늘 쓴 `*` 와 `shortestPath()` 도 **그대로 동작하지만** 문서에는 표준을 따르지 않는 옛 표기로 적혀 있습니다.

| 오늘 배운 표기 | 문서에 나오는 새 표기 |
|---|---|
| `-[:NEXT_TO*1..3]-` | `-[:NEXT_TO]-{1,3}` (수량자) |
| `(a)-[:NEXT_TO*1..3]-(b)` | `((a)-[:NEXT_TO]-(b)){1,3}` (수량자 패턴) |
| `shortestPath( ... )` | `SHORTEST 1 ( ... )` |
| `allShortestPaths( ... )` | `ALL SHORTEST ( ... )` |
| (닿는지만 볼 때) `shortestPath` | `ANY ( ... )` |

**우리 수업은 `*` 와 `shortestPath` 로 통일합니다.** 뒤 단원(day32 이후)과 시중 자료 대부분이 이 표기를 쓰고, 검색해서 나오는 예제도 이쪽이 훨씬 많기 때문입니다. 다만 **공식 문서를 읽을 때 낯선 중괄호 표기가 나와도 같은 일을 하는 다른 표기**라는 것만 알아 두세요.

---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1-1 | `-[:R*1..2]-` + `DISTINCT` | 관계를 1~2 칸 이어서 따라감. 갈래가 여럿이면 같은 노드가 여러 번 나온다 |
| 1-1 | `*2..2` / `*..2` / `*1..` / `*` / `*0..2` | 하한을 안 적으면 1. 하한 0 은 자기 자신을 답에 넣는다 |
| 2-1 | `shortestPath( ... )` | 최단 경로를 **한 갈래** 돌려준다. 경로 변수 `p =` 로 담는다 |
| 2-2 | `allShortestPaths( ... )` | **가장 짧은 길이**의 경로를 전부 돌려준다. 하나만 받고 유일하다고 말하지 말 것 |
| 3-1 | `length(p)` / `nodes(p)` / `relationships(p)` | 이동 수 / 지나는 노드 목록 / 탄 관계 목록. length = 노드 수 − 1 |
| 3-2 | `[n IN nodes(p) \| n.name]` | 목록을 훑어 값만 뽑는다. 세로선 앞에 `WHERE` 를 붙이면 걸러 뽑는다 |
| 3-3 | `ORDER BY length(p) DESC, 이름` + `LIMIT` | 거리로 줄 세운다. 동점 대비 보조 정렬키, 많으면 잘라 받는다 |
| 3-4 | `all` / `any` / `none` | 경로가 지난 구간을 전부 훑어 조건을 건다 |

- 방향 없는 관계(전철 인접)는 조회할 때 화살표 없이 `-[:NEXT_TO]-` 로 쓰고, 방향이 뜻을 가지는 관계(송금)는 화살표를 살려 `-[:TRANSFER*]->` 로 씁니다.
- "몇 칸 안"을 물으면 가변길이, "어떻게 가나"를 물으면 `shortestPath` 입니다. 가변길이는 **한쪽 끝에 이름을 박고** 쓰세요. 도착 노드 이름만 받으면 상한을 열어 둬도 빠르지만, **경로 값을 함께 꺼내면** 상한을 두거나 `shortestPath` 로 감싸야 합니다. 이 659역 그래프에서 서울역 기준 `*1..12` 는 경로가 133,425갈래입니다.
- **최단 경로는 하나가 아닐 수 있습니다.** 여럿일 때 무엇을 고를지는 데이터가 아니라 우리가 정합니다.
- 닿지 못하는 노드는 **행 자체가 생기지 않습니다.** 없는 것까지 세려면 다음 시간의 `OPTIONAL MATCH` 가 필요합니다.
- `length(p)` 는 정렬에는 그대로 쓰지만, **거르는 조건**으로 쓰려면 다음 시간의 `WITH` 가 필요합니다.

## ⏭️ 예고: 다음 시간

경로를 찾았으니, 이제 **여러 조건으로 거르고 다듬는** 법을 배웁니다. `IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH`·정규식으로 세밀하게 필터하고 그것들을 `AND`·`OR` 로 묶습니다. **관계가 있는지 없는지 자체를 조건으로** 쓰고, **`OPTIONAL MATCH`** 로 없는 것도 빠뜨리지 않고, **`WITH`** 파이프라인으로 중간 결과를 단계별로 넘기며, `ORDER BY`·`LIMIT`·`SKIP` 로 정렬해 상위만 뽑고 그다음 쪽으로 넘깁니다.

수고하셨습니다!